## Initial Setup

In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")

## Question 1

Check which of the listed columns contains missing values.

In [10]:
df[['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']].isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
dtype: int64

### Answer

`horsepower`

## Question 2

What's the median (50% percentile) for variable `horsepower`?

In [19]:
df.horsepower.median()

np.float64(254.0)

### Answer

254

## Prepare and split the dataset

In [12]:
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]

df = df[features]

In [13]:
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [14]:
df.shape

(10000, 5)

In [15]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [17]:
print(len(df_train))
print(len(df_val))
print(len(df_test))

6000
2000
2000


## Question 3

We compare two ways of handling missing values in `horsepower`:

1. Fill missing values with `0`
2. Fill missing values with the mean of `horsepower` calculated from the training dataset only

We train an unregularized linear regression model for each option and compare validation RMSE.

In [20]:
target = 'fuel_efficiency_mpg'

features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

X_train = df_train[features].copy()
y_train = df_train[target].copy()

X_val = df_val[features].copy()
y_val = df_val[target].copy()

In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

print("scikit-learn is working")

scikit-learn is working


In [24]:
X_train_zero = X_train.fillna(0)
X_val_zero = X_val.fillna(0)

model_zero = LinearRegression()
model_zero.fit(X_train_zero, y_train)

y_pred_zero = model_zero.predict(X_val_zero)

rmse_zero = root_mean_squared_error(y_val, y_pred_zero)

round(rmse_zero, 3)

2.205

In [25]:
horsepower_mean = X_train['horsepower'].mean()

X_train_mean = X_train.fillna(horsepower_mean)
X_val_mean = X_val.fillna(horsepower_mean)

model_mean = LinearRegression()
model_mean.fit(X_train_mean, y_train)

y_pred_mean = model_mean.predict(X_val_mean)

rmse_mean = root_mean_squared_error(y_val, y_pred_mean)

round(rmse_mean, 3)

2.202

In [26]:
print("RMSE with 0:    ", round(rmse_zero, 3))
print("RMSE with mean: ", round(rmse_mean, 3))

RMSE with 0:     2.205
RMSE with mean:  2.202


### Answer

The option with the lower RMSE gives the better result.

- With 0: [result]
- With mean: [result]

Therefore, the answer is: **[Both are equally good]**.

## Question 4

Train a regularized linear regression model.

Missing values in `horsepower` are filled with `0`. We try different regularization values `r` and evaluate each model using validation RMSE.

In [46]:
def train_linear_regression_reg(X, y, r=0.0):
    X = X.to_numpy(dtype=float)

    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T @ X

    regularizer = r * np.eye(XTX.shape[0])
    regularizer[0, 0] = 0.0

    w = np.linalg.solve(
        XTX + regularizer,
        X.T @ y.to_numpy(dtype=float)
    )

    return w

In [47]:
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

X_train = df_train[features].fillna(0)
y_train = df_train[target]

X_val = df_val[features].fillna(0)
y_val = df_val[target]

In [48]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

results = {}

for r in r_values:
    w = train_linear_regression_reg(X_train, y_train, r)

    X_val_matrix = np.column_stack([
        np.ones(len(X_val)),
        X_val.to_numpy(dtype=float)
    ])

    y_pred = X_val_matrix @ w

    rmse = np.sqrt(np.mean((y_val.to_numpy() - y_pred) ** 2))

    results[r] = rmse

In [49]:
for r in r_values:
    print(f"r = {r:<6} RMSE = {results[r]:.4f}")

r = 0      RMSE = 2.2070
r = 0.01   RMSE = 2.2070
r = 0.1    RMSE = 2.2070
r = 1      RMSE = 2.2070
r = 5      RMSE = 2.2070
r = 10     RMSE = 2.2070
r = 100    RMSE = 2.2070


In [50]:
best_r = min(
    r_values,
    key=lambda r: (round(results[r], 4), r)
)

print("Best r:", best_r)
print("Best RMSE:", round(results[best_r], 4))

Best r: 0
Best RMSE: 2.207


### Answer

All tested values of `r` produce the same RMSE when rounded to 4 decimal places: `2.2070`.

Following the instruction to choose the smallest `r` when there is a tie, the answer is **0**.

## Question 5

The random seed affects how the dataset is split into training, validation, and test sets.

We repeat the same 60/20/20 split for seeds 0 through 9. For each split, we fill missing `horsepower` values with `0`, train an unregularized linear regression model, and calculate the validation RMSE.

Finally, we calculate the standard deviation of the 10 RMSE scores to measure how sensitive the result is to the random seed.

In [30]:
scores = []

features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

for seed in range(10):
    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train_seed = df.iloc[idx[:n_train]]
    df_val_seed = df.iloc[idx[n_train:n_train + n_val]]

    X_train_seed = df_train_seed[features].fillna(0)
    y_train_seed = df_train_seed[target]

    X_val_seed = df_val_seed[features].fillna(0)
    y_val_seed = df_val_seed[target]

    model = LinearRegression()
    model.fit(X_train_seed, y_train_seed)

    y_pred = model.predict(X_val_seed)

    rmse = root_mean_squared_error(y_val_seed, y_pred)

    scores.append(rmse)

In [31]:
for seed, score in enumerate(scores):
    print(f"seed = {seed}: RMSE = {score:.4f}")

seed = 0: RMSE = 2.2392
seed = 1: RMSE = 2.2021
seed = 2: RMSE = 2.1634
seed = 3: RMSE = 2.2044
seed = 4: RMSE = 2.2019
seed = 5: RMSE = 2.2458
seed = 6: RMSE = 2.2697
seed = 7: RMSE = 2.1908
seed = 8: RMSE = 2.2166
seed = 9: RMSE = 2.2070


In [32]:
std_score = np.std(scores)

print("Standard deviation:", round(std_score, 3))

Standard deviation: 0.029


### Answer

The standard deviation of the validation RMSE scores is **0.029**.

Therefore, the answer is **0.029**.

## Question 6

Using seed `9`, split the dataset into 60% training, 20% validation, and 20% test.

Combine the training and validation datasets, fill missing values with `0`, train a regularized linear regression model with `r = 0.001`, and evaluate it on the test dataset using RMSE.

In [58]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

df_train_9 = df.iloc[idx[:n_train]]
df_val_9 = df.iloc[n_train:n_train + n_val]
df_test_9 = df.iloc[n_train + n_val:]

In [59]:
print(len(df_train_9))
print(len(df_val_9))
print(len(df_test_9))

6000
2000
2000


In [60]:
df_train_full = pd.concat([
    df_train_9,
    df_val_9
])

print(len(df_train_full))

8000


In [61]:
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

X_train_full = df_train_full[features].fillna(0)
y_train_full = df_train_full[target]

X_test = df_test_9[features].fillna(0)
y_test = df_test_9[target]

In [62]:
w = train_linear_regression_reg(
    X_train_full,
    y_train_full,
    r=0.001
)

In [63]:
X_test_matrix = np.column_stack([
    np.ones(len(X_test)),
    X_test.to_numpy(dtype=float)
])

y_pred = X_test_matrix @ w

In [64]:
rmse_test = np.sqrt(
    np.mean(
        (y_test.to_numpy() - y_pred) ** 2
    )
)

print("Test RMSE:", rmse_test)
print("Rounded Test RMSE:", round(rmse_test, 3))

Test RMSE: 2.2775888434918823
Rounded Test RMSE: 2.278


### Answer

The RMSE on the test dataset is **2.278**.